Inspect BDD100K Dataset Structure

In [3]:
from pathlib import Path

PROJECT_ROOT = Path(r"E:\ML_Project")
DATASET_ROOT = (
    PROJECT_ROOT / "datasets" / "bdd100k_original_yolo"
)

print("=" * 75)
print("BDD100K DATASET STRUCTURE")
print("=" * 75)
print(f"\nDataset root:")
print(DATASET_ROOT)
print("\nExists:", DATASET_ROOT.exists())

if DATASET_ROOT.exists():

    print("\nTop-level contents")
    print("-" * 75)

    for item in sorted(DATASET_ROOT.iterdir()):
        item_type = (
            "[DIR] "
            if item.is_dir()
            else "[FILE]"
        )
        print(
            f"{item_type} {item.name}"
        )
else:

    print("\n❌ Dataset folder was not found.")

BDD100K DATASET STRUCTURE

Dataset root:
E:\ML_Project\datasets\bdd100k_original_yolo

Exists: True

Top-level contents
---------------------------------------------------------------------------
[FILE] bdd100k.yaml
[DIR]  test
[DIR]  train
[DIR]  valid


Checking BDD100K Image and Label Folders

In [4]:
from pathlib import Path

print("=" * 75)
print("BDD100K IMAGE / LABEL STRUCTURE")
print("=" * 75)

important_dirs = [
    DATASET_ROOT / "train",
    DATASET_ROOT / "valid",
    DATASET_ROOT / "val",
    DATASET_ROOT / "test",
    DATASET_ROOT / "images",
    DATASET_ROOT / "labels",
]

print("\nChecking expected directories")
print("-" * 75)

for path in important_dirs:
    if path.exists():
        print(f"✓ {path}")
        if path.is_dir():
            children = list(path.iterdir())
            print(
                f"    └── {len(children):,} item(s)"
            )
    else:
        print(f"✗ {path}")

print("\n" + "=" * 75)
print("IMAGE / LABEL DIRECTORIES FOUND")
print("=" * 75)

image_dirs = []
label_dirs = []

for path in DATASET_ROOT.rglob("*"):
    if path.is_dir():
        name = path.name.lower()
        if name == "images":
            image_dirs.append(path)
        elif name == "labels":
            label_dirs.append(path)

print("\nImage directories:")
for path in image_dirs:
    print(f"  {path}")
print("\nLabel directories:")
for path in label_dirs:
    print(f"  {path}")
print("\n" + "=" * 75)
print("STRUCTURE INSPECTION COMPLETE")
print("=" * 75)

BDD100K IMAGE / LABEL STRUCTURE

Checking expected directories
---------------------------------------------------------------------------
✓ E:\ML_Project\datasets\bdd100k_original_yolo\train
    └── 2 item(s)
✓ E:\ML_Project\datasets\bdd100k_original_yolo\valid
    └── 2 item(s)
✗ E:\ML_Project\datasets\bdd100k_original_yolo\val
✓ E:\ML_Project\datasets\bdd100k_original_yolo\test
    └── 2 item(s)
✗ E:\ML_Project\datasets\bdd100k_original_yolo\images
✗ E:\ML_Project\datasets\bdd100k_original_yolo\labels

IMAGE / LABEL DIRECTORIES FOUND

Image directories:
  E:\ML_Project\datasets\bdd100k_original_yolo\test\images
  E:\ML_Project\datasets\bdd100k_original_yolo\train\images
  E:\ML_Project\datasets\bdd100k_original_yolo\valid\images

Label directories:
  E:\ML_Project\datasets\bdd100k_original_yolo\test\labels
  E:\ML_Project\datasets\bdd100k_original_yolo\train\labels
  E:\ML_Project\datasets\bdd100k_original_yolo\valid\labels

STRUCTURE INSPECTION COMPLETE


Verify BDD100K Images and YOLO Labels

In [5]:
from pathlib import Path
from collections import Counter

print("=" * 75)
print("BDD100K IMAGE AND LABEL VERIFICATION")
print("=" * 75)

TRAIN_IMAGES = DATASET_ROOT / "train" / "images"
TRAIN_LABELS = DATASET_ROOT / "train" / "labels"

VALID_IMAGES = DATASET_ROOT / "valid" / "images"
VALID_LABELS = DATASET_ROOT / "valid" / "labels"

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp"
}

def get_image_files(folder):

    return sorted([
        p for p in folder.iterdir()
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTENSIONS
    ])

def get_label_files(folder):

    return sorted([
        p for p in folder.iterdir()
        if p.is_file()
        and p.suffix.lower() == ".txt"
    ])

train_images = get_image_files(
    TRAIN_IMAGES
)
train_labels = get_label_files(
    TRAIN_LABELS
)
valid_images = get_image_files(
    VALID_IMAGES
)
valid_labels = get_label_files(
    VALID_LABELS
)

print("\nTRAIN")
print("-" * 75)
print(
    f"Images : {len(train_images):,}"
)
print(
    f"Labels : {len(train_labels):,}"
)
print("\nVALID")
print("-" * 75)
print(
    f"Images : {len(valid_images):,}"
)
print(
    f"Labels : {len(valid_labels):,}"
)

train_image_stems = {
    p.stem for p in train_images
}
train_label_stems = {
    p.stem for p in train_labels
}
valid_image_stems = {
    p.stem for p in valid_images
}
valid_label_stems = {
    p.stem for p in valid_labels
}
train_missing_labels = (
    train_image_stems -
    train_label_stems
)
train_orphan_labels = (
    train_label_stems -
    train_image_stems
)
valid_missing_labels = (
    valid_image_stems -
    valid_label_stems
)
valid_orphan_labels = (
    valid_label_stems -
    valid_image_stems
)
print("\nFILE MATCHING")
print("-" * 75)
print(
    f"Train images without labels : "
    f"{len(train_missing_labels):,}"
)
print(
    f"Train orphan labels         : "
    f"{len(train_orphan_labels):,}"
)
print(
    f"Valid images without labels : "
    f"{len(valid_missing_labels):,}"
)
print(
    f"Valid orphan labels         : "
    f"{len(valid_orphan_labels):,}"
)

CLASS_NAMES = [
    "bike",
    "bus",
    "car",
    "motor",
    "person",
    "rider",
    "traffic light",
    "traffic sign",
    "train",
    "truck"
]


class_counter = Counter()
invalid_label_files = []
invalid_label_lines = []
all_label_files = (
    train_labels +
    valid_labels
)
for label_file in all_label_files:
    try:
        with open(
            label_file,
            "r",
            encoding="utf-8"
        ) as f:
            lines = [
                line.strip()
                for line in f
                if line.strip()
            ]
        for line_number, line in enumerate(
            lines,
            start=1
        ):
            parts = line.split()

            # YOLO format:
            # class x_center y_center width height

            if len(parts) != 5:

                invalid_label_lines.append(
                    (
                        label_file,
                        line_number,
                        line
                    )
                )

                continue


            try:

                class_id = int(
                    parts[0]
                )

                values = [
                    float(x)
                    for x in parts[1:]
                ]

            except ValueError:

                invalid_label_lines.append(
                    (
                        label_file,
                        line_number,
                        line
                    )
                )

                continue


            # Check class ID

            if (
                class_id < 0
                or class_id >= len(CLASS_NAMES)
            ):

                invalid_label_lines.append(
                    (
                        label_file,
                        line_number,
                        line
                    )
                )

                continue


            # Check bounding box values

            if not all(
                0.0 <= x <= 1.0
                for x in values
            ):

                invalid_label_lines.append(
                    (
                        label_file,
                        line_number,
                        line
                    )
                )

                continue


            class_counter[class_id] += 1


    except Exception:

        invalid_label_files.append(
            label_file
        )

print("\nCLASS DISTRIBUTION")
print("-" * 75)

for class_id in range(
    len(CLASS_NAMES)
):
    count = class_counter[class_id]
    print(
        f"{class_id:2d} | "
        f"{CLASS_NAMES[class_id]:15s} | "
        f"{count:,}"
    )

print("\n" + "=" * 75)
print("VALIDATION SUMMARY")
print("=" * 75)
print(
    f"✓ Train images       : {len(train_images):,}"
)
print(
    f"✓ Train labels       : {len(train_labels):,}"
)
print(
    f"✓ Valid images       : {len(valid_images):,}"
)
print(
    f"✓ Valid labels       : {len(valid_labels):,}"
)
print(
    f"✓ Invalid label files: "
    f"{len(invalid_label_files):,}"
)
print(
    f"✓ Invalid label lines: "
    f"{len(invalid_label_lines):,}"
)
if (
    len(train_missing_labels) == 0
    and len(train_orphan_labels) == 0
    and len(valid_missing_labels) == 0
    and len(valid_orphan_labels) == 0
    and len(invalid_label_files) == 0
    and len(invalid_label_lines) == 0
):
    print("\nSTATUS: ✓ DATASET STRUCTURE AND LABELS LOOK VALID")

else:
    print(
        "\nSTATUS: ⚠ DATASET ISSUES DETECTED — "
        "DO NOT TRAIN YET"
    )

BDD100K IMAGE AND LABEL VERIFICATION

TRAIN
---------------------------------------------------------------------------
Images : 70,000
Labels : 70,000

VALID
---------------------------------------------------------------------------
Images : 10,000
Labels : 10,000

FILE MATCHING
---------------------------------------------------------------------------
Train images without labels : 0
Train orphan labels         : 0
Valid images without labels : 0
Valid orphan labels         : 0

CLASS DISTRIBUTION
---------------------------------------------------------------------------
 0 | bike            | 8,232
 1 | bus             | 13,281
 2 | car             | 816,423
 3 | motor           | 3,454
 4 | person          | 104,667
 5 | rider           | 5,170
 6 | traffic light   | 213,109
 7 | traffic sign    | 274,801
 8 | train           | 151
 9 | truck           | 34,248

VALIDATION SUMMARY
✓ Train images       : 70,000
✓ Train labels       : 70,000
✓ Valid images       : 10,000
✓ Valid la

Inspect Raw YOLO Label Contents

In [6]:
print("=" * 75)
print("RAW BDD100K LABEL INSPECTION")
print("=" * 75)

sample_label_files = train_labels[:3]

for i, label_file in enumerate(
    sample_label_files,
    start=1
):

    print("\n" + "-" * 75)
    print(f"LABEL FILE {i}")
    print("-" * 75)
    print(
        f"File: {label_file.name}"
    )

    with open(
        label_file,
        "r",
        encoding="utf-8"
    ) as f:

        lines = [
            line.rstrip("\n")
            for line in f
            if line.strip()
        ]

    print(
        f"Number of annotation lines: {len(lines)}"
    )
    print("\nFirst 5 raw lines:")

    for line in lines[:5]:
        print(
            repr(line)
        )

print("\n" + "=" * 75)
print("RAW LABEL INSPECTION COMPLETE")
print("=" * 75)

RAW BDD100K LABEL INSPECTION

---------------------------------------------------------------------------
LABEL FILE 1
---------------------------------------------------------------------------
File: 0000f77c-6257be58.txt
Number of annotation lines: 7

First 5 raw lines:
'6 0.891750 0.238931 0.024278 0.107904'
'6 0.917378 0.241328 0.026976 0.103108'
'7 0.887704 0.308811 0.053952 0.031172'
'7 0.039212 0.085467 0.078423 0.170249'
'2 0.157440 0.515581 0.244191 0.324133'

---------------------------------------------------------------------------
LABEL FILE 2
---------------------------------------------------------------------------
File: 0000f77c-62c2a288.txt
Number of annotation lines: 6

First 5 raw lines:
'7 0.218217 0.452551 0.044950 0.086713'
'7 0.167050 0.431297 0.011477 0.030605'
'7 0.727177 0.406077 0.009564 0.015302'
'4 0.334897 0.482305 0.008608 0.030605'
'4 0.444404 0.467003 0.007651 0.034005'

---------------------------------------------------------------------------
LABEL 

VALIDATE STANDARD YOLO ANNOTATIONS

In [8]:
from pathlib import Path
from collections import Counter

print("=" * 75)
print("VALIDATING BDD100K ORIGIN — STANDARD YOLO ANNOTATIONS")
print("=" * 75)

PROJECT_ROOT = Path(r"E:\ML_Project")
DATASET_ROOT = PROJECT_ROOT / "datasets" / "bdd100k_original_yolo"

TRAIN_IMAGES = DATASET_ROOT / "train" / "images"
TRAIN_LABELS = DATASET_ROOT / "train" / "labels"

VALID_IMAGES = DATASET_ROOT / "valid" / "images"
VALID_LABELS = DATASET_ROOT / "valid" / "labels"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

CLASS_NAMES = [
    "bike",
    "bus",
    "car",
    "motor",
    "person",
    "rider",
    "traffic light",
    "traffic sign",
    "train",
    "truck"
]

NUM_CLASSES = len(CLASS_NAMES)

def get_image_stems(folder):
    return {
        p.stem
        for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    }

def get_label_stems(folder):
    return {
        p.stem
        for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() == ".txt"
    }

def validate_labels(label_folder):
    invalid_lines = []
    out_of_range = []
    invalid_class_ids = []
    zero_or_negative_boxes = []
    class_counter = Counter()
    total_objects = 0
    label_files = sorted(label_folder.glob("*.txt"))
    for label_file in label_files:
        with open(label_file, "r", encoding="utf-8") as f:
            lines = [line.strip() for line in f if line.strip()]
        for line_number, line in enumerate(lines, start=1):
            parts = line.split()

            # Standard YOLO format:
            # class_id x_center y_center width height
            if len(parts) != 5:
                invalid_lines.append(
                    (label_file.name, line_number, len(parts), line)
                )
                continue
            try:
                class_id = int(parts[0])
                x_center = float(parts[1])
                y_center = float(parts[2])
                width = float(parts[3])
                height = float(parts[4])
            except ValueError:
                invalid_lines.append(
                    (label_file.name, line_number, "non-numeric", line)
                )
                continue
            total_objects += 1
            class_counter[class_id] += 1
            if not (0 <= class_id < NUM_CLASSES):
                invalid_class_ids.append(
                    (label_file.name, line_number, class_id)
                )
            coordinates = [
                x_center,
                y_center,
                width,
                height
            ]
            if not all(0.0 <= value <= 1.0 for value in coordinates):
                out_of_range.append(
                    (label_file.name, line_number, coordinates)
                )
            # Box size validation
            if width <= 0 or height <= 0:
                zero_or_negative_boxes.append(
                    (label_file.name, line_number, width, height)
                )
    return {
        "label_files": len(label_files),
        "total_objects": total_objects,
        "class_counter": class_counter,
        "invalid_lines": invalid_lines,
        "invalid_class_ids": invalid_class_ids,
        "out_of_range": out_of_range,
        "zero_or_negative_boxes": zero_or_negative_boxes,
    }

required_directories = [
    TRAIN_IMAGES,
    TRAIN_LABELS,
    VALID_IMAGES,
    VALID_LABELS
]

print("\nDataset path")
print("-" * 75)
print(DATASET_ROOT)
print("\nChecking required directories")
print("-" * 75)

for directory in required_directories:
    print(f"{directory}: {'OK' if directory.exists() else 'MISSING'}")

train_image_stems = get_image_stems(TRAIN_IMAGES)
train_label_stems = get_label_stems(TRAIN_LABELS)

valid_image_stems = get_image_stems(VALID_IMAGES)
valid_label_stems = get_label_stems(VALID_LABELS)

train_images_without_labels = train_image_stems - train_label_stems
train_labels_without_images = train_label_stems - train_image_stems

valid_images_without_labels = valid_image_stems - valid_label_stems
valid_labels_without_images = valid_label_stems - valid_image_stems

print("\nTRAIN SET")
print("-" * 75)
print(f"Images : {len(train_image_stems):,}")
print(f"Labels : {len(train_label_stems):,}")
print(f"Images without labels : {len(train_images_without_labels):,}")
print(f"Labels without images : {len(train_labels_without_images):,}")
print("\nVALID SET")
print("-" * 75)
print(f"Images : {len(valid_image_stems):,}")
print(f"Labels : {len(valid_label_stems):,}")
print(f"Images without labels : {len(valid_images_without_labels):,}")
print(f"Labels without images : {len(valid_labels_without_images):,}")
print("\nValidating TRAIN labels...")
train_results = validate_labels(TRAIN_LABELS)
print("Validating VALID labels...")
valid_results = validate_labels(VALID_LABELS)
print("\n" + "=" * 75)
print("ANNOTATION VALIDATION SUMMARY")
print("=" * 75)
print("\nTRAIN")
print("-" * 75)
print(f"Label files           : {train_results['label_files']:,}")
print(f"Total objects         : {train_results['total_objects']:,}")
print(f"Invalid lines         : {len(train_results['invalid_lines']):,}")
print(f"Invalid class IDs     : {len(train_results['invalid_class_ids']):,}")
print(f"Out-of-range values   : {len(train_results['out_of_range']):,}")
print(
    f"Invalid box dimensions: "
    f"{len(train_results['zero_or_negative_boxes']):,}"
)
print("\nVALID")
print("-" * 75)
print(f"Label files           : {valid_results['label_files']:,}")
print(f"Total objects         : {valid_results['total_objects']:,}")
print(f"Invalid lines         : {len(valid_results['invalid_lines']):,}")
print(f"Invalid class IDs     : {len(valid_results['invalid_class_ids']):,}")
print(f"Out-of-range values   : {len(valid_results['out_of_range']):,}")
print(
    f"Invalid box dimensions: "
    f"{len(valid_results['zero_or_negative_boxes']):,}"
)
print("\nTRAIN CLASS DISTRIBUTION")
print("-" * 75)

for class_id in range(NUM_CLASSES):
    count = train_results["class_counter"].get(class_id, 0)
print(f"{class_id:2d} | {CLASS_NAMES[class_id]:15s} | {count:,}")
print("\nVALID CLASS DISTRIBUTION")
print("-" * 75)

for class_id in range(NUM_CLASSES):
    count = valid_results["class_counter"].get(class_id, 0)
    print(f"{class_id:2d} | {CLASS_NAMES[class_id]:15s} | {count:,}")

dataset_is_valid = (
    TRAIN_IMAGES.exists()
    and TRAIN_LABELS.exists()
    and VALID_IMAGES.exists()
    and VALID_LABELS.exists()
    and len(train_image_stems) == len(train_label_stems)
    and len(valid_image_stems) == len(valid_label_stems)
    and len(train_images_without_labels) == 0
    and len(train_labels_without_images) == 0
    and len(valid_images_without_labels) == 0
    and len(valid_labels_without_images) == 0
    and len(train_results["invalid_lines"]) == 0
    and len(valid_results["invalid_lines"]) == 0
    and len(train_results["invalid_class_ids"]) == 0
    and len(valid_results["invalid_class_ids"]) == 0
    and len(train_results["out_of_range"]) == 0
    and len(valid_results["out_of_range"]) == 0
    and len(train_results["zero_or_negative_boxes"]) == 0
    and len(valid_results["zero_or_negative_boxes"]) == 0
)

print("\n" + "=" * 75)

if dataset_is_valid:
    print("STATUS: PASS")
    print("BDD100K origin is valid standard YOLO format.")
else:
    print("STATUS: CHECK REQUIRED")
    print("One or more validation checks failed.")

print("=" * 75)

VALIDATING BDD100K ORIGIN — STANDARD YOLO ANNOTATIONS

Dataset path
---------------------------------------------------------------------------
E:\ML_Project\datasets\bdd100k_original_yolo

Checking required directories
---------------------------------------------------------------------------
E:\ML_Project\datasets\bdd100k_original_yolo\train\images: OK
E:\ML_Project\datasets\bdd100k_original_yolo\train\labels: OK
E:\ML_Project\datasets\bdd100k_original_yolo\valid\images: OK
E:\ML_Project\datasets\bdd100k_original_yolo\valid\labels: OK

TRAIN SET
---------------------------------------------------------------------------
Images : 70,000
Labels : 70,000
Images without labels : 0
Labels without images : 0

VALID SET
---------------------------------------------------------------------------
Images : 10,000
Labels : 10,000
Images without labels : 0
Labels without images : 0

Validating TRAIN labels...
Validating VALID labels...

ANNOTATION VALIDATION SUMMARY

TRAIN
---------------------

CREATE YOLO26 DATASET CONFIGURATION

In [9]:
from pathlib import Path
import yaml

print("=" * 75)
print("CREATING YOLO26 DATASET CONFIGURATION")
print("=" * 75)

PROJECT_ROOT = Path(r"E:\ML_Project")
DATASET_ROOT = PROJECT_ROOT / "datasets" / "bdd100k_original_yolo"

TRAIN_IMAGES = DATASET_ROOT / "train" / "images"
VALID_IMAGES = DATASET_ROOT / "valid" / "images"

RESULTS_PATH = PROJECT_ROOT / "results"
MODELS_PATH = PROJECT_ROOT / "models"

RESULTS_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = [
    "bike",
    "bus",
    "car",
    "motor",
    "person",
    "rider",
    "traffic light",
    "traffic sign",
    "train",
    "truck"
]

DATA_YAML = RESULTS_PATH / "bdd100k_origin.yaml"
data_config = {
    "path": str(DATASET_ROOT),
    "train": "train/images",
    "val": "valid/images",
    "names": {
        i: name
        for i, name in enumerate(CLASS_NAMES)
    }
}
with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(
        data_config,
        f,
        sort_keys=False,
        allow_unicode=True
    )

print("\nDataset root")
print("-" * 75)
print(DATASET_ROOT)
print("\nTrain images")
print("-" * 75)
print(TRAIN_IMAGES)
print("\nValidation images")
print("-" * 75)
print(VALID_IMAGES)
print("\nYAML file")
print("-" * 75)
print(DATA_YAML)
print("\nConfiguration")
print("-" * 75)
with open(DATA_YAML, "r", encoding="utf-8") as f:
    print(f.read())

print("=" * 75)

if TRAIN_IMAGES.exists() and VALID_IMAGES.exists() and DATA_YAML.exists():
    print("STATUS: PASS")
    print("YOLO26 dataset configuration created successfully.")
else:
    print("STATUS: CHECK REQUIRED")

print("=" * 75)

CREATING YOLO26 DATASET CONFIGURATION

Dataset root
---------------------------------------------------------------------------
E:\ML_Project\datasets\bdd100k_original_yolo

Train images
---------------------------------------------------------------------------
E:\ML_Project\datasets\bdd100k_original_yolo\train\images

Validation images
---------------------------------------------------------------------------
E:\ML_Project\datasets\bdd100k_original_yolo\valid\images

YAML file
---------------------------------------------------------------------------
E:\ML_Project\results\bdd100k_origin.yaml

Configuration
---------------------------------------------------------------------------
path: E:\ML_Project\datasets\bdd100k_original_yolo
train: train/images
val: valid/images
names:
  0: bike
  1: bus
  2: car
  3: motor
  4: person
  5: rider
  6: traffic light
  7: traffic sign
  8: train
  9: truck

STATUS: PASS
YOLO26 dataset configuration created successfully.


verify the YOLO26 training environment

In [10]:
import sys
import torch

print("=" * 75)
print("YOLO26 TRAINING ENVIRONMENT CHECK")
print("=" * 75)
print("\nPYTHON")
print("-" * 75)
print("Python executable :", sys.executable)
print("Python version    :", sys.version.split()[0])
print("\nPYTORCH")
print("-" * 75)
print("PyTorch version   :", torch.__version__)
print("CUDA available    :", torch.cuda.is_available())
print("CUDA version      :", torch.version.cuda)

if torch.cuda.is_available():

    print("\nGPU")
    print("-" * 75)
    gpu_count = torch.cuda.device_count()
    print("GPU count         :", gpu_count)
    for i in range(gpu_count):
        gpu_name = torch.cuda.get_device_name(i)
        gpu_memory = torch.cuda.get_device_properties(i).total_memory
        print(f"\nGPU {i}")
        print("Name              :", gpu_name)
        print(
            "Total VRAM        :",
            f"{gpu_memory / (1024 ** 3):.2f} GB"
        )
else:
    print("\nGPU")
    print("-" * 75)
    print("CUDA GPU not available.")

print("\nULTRALYTICS")
print("-" * 75)

try:
    import ultralytics

    print("Ultralytics version:", ultralytics.__version__)
    print("Ultralytics path   :", ultralytics.__file__)

except ImportError:
    print("Ultralytics is NOT installed.")

print("\n" + "=" * 75)
    
if torch.cuda.is_available():
    print("STATUS: GPU READY")
else:
    print("STATUS: CUDA GPU NOT AVAILABLE")

print("=" * 75)

YOLO26 TRAINING ENVIRONMENT CHECK

PYTHON
---------------------------------------------------------------------------
Python executable : D:\Software Apps\Python\python.exe
Python version    : 3.10.11

PYTORCH
---------------------------------------------------------------------------
PyTorch version   : 2.12.0.dev20260226+cu128
CUDA available    : True
CUDA version      : 12.8

GPU
---------------------------------------------------------------------------
GPU count         : 1

GPU 0
Name              : NVIDIA GeForce RTX 5060 Ti
Total VRAM        : 15.93 GB

ULTRALYTICS
---------------------------------------------------------------------------
Ultralytics version: 8.4.18
Ultralytics path   : D:\Software Apps\Python\lib\site-packages\ultralytics\__init__.py

STATUS: GPU READY


Load and verify YOLO26

In [11]:
from pathlib import Path
from ultralytics import YOLO
import torch

print("=" * 75)
print("YOLO26 MODEL VERIFICATION")
print("=" * 75)

PROJECT_ROOT = Path(r"E:\ML_Project")
MODELS_PATH = PROJECT_ROOT / "models"
RESULTS_PATH = PROJECT_ROOT / "results"

MODELS_PATH.mkdir(parents=True, exist_ok=True)
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# YOLO26 model
MODEL_NAME = "yolo26n.pt"

print("\nModel")
print("-" * 75)
print("Requested model :", MODEL_NAME)

# Load pretrained YOLO26 model
model = YOLO(MODEL_NAME)

print("\nModel loaded successfully.")
print("Model type      :", type(model).__name__)
print("\nModel information")
print("-" * 75)

try:
    model.info()
except Exception as e:
    print("Model info could not be displayed:")
    print(e)
print("\nDevice")
print("-" * 75)

if torch.cuda.is_available():
    print("CUDA available : True")
    print("GPU            :", torch.cuda.get_device_name(0))
else:
    print("CUDA available : False")
    print("GPU            : CPU")

print("\n" + "=" * 75)
print("STATUS: YOLO26 MODEL LOADED")
print("=" * 75)

YOLO26 MODEL VERIFICATION

Model
---------------------------------------------------------------------------
Requested model : yolo26n.pt

Model loaded successfully.
Model type      : YOLO

Model information
---------------------------------------------------------------------------
YOLO26n summary: 260 layers, 2,572,280 parameters, 0 gradients, 6.1 GFLOPs

Device
---------------------------------------------------------------------------
CUDA available : True
GPU            : NVIDIA GeForce RTX 5060 Ti

STATUS: YOLO26 MODEL LOADED


train the real baseline by yolo26 for 30 epoches with 10 patience

In [14]:
from pathlib import Path
from ultralytics import YOLO
import torch

PROJECT_ROOT = Path(r"E:\ML_Project")
DATASET_ROOT = (
    PROJECT_ROOT
    / "datasets"
    / "bdd100k_original_yolo"
)

RESULTS_ROOT = (
    PROJECT_ROOT
    / "results"
)
BASELINE_RUNS = (
    RESULTS_ROOT
    / "yolo26_baseline"
)


BASELINE_RUNS.mkdir(
    parents=True,
    exist_ok=True
)


print("=" * 75)
print("YOLO26n BASELINE TRAINING")
print("=" * 75)
print("\nProject root:")
print(PROJECT_ROOT)
print("\nDataset:")
print(DATASET_ROOT)
print("\nResults directory:")
print(BASELINE_RUNS)
print("\n" + "-" * 75)
print("DATASET CHECK")
print("-" * 75)

if not DATASET_ROOT.exists():
    raise FileNotFoundError(
        f"\nDataset directory was not found:\n{DATASET_ROOT}"
    )

print("Dataset directory : FOUND")


DATA_YAML = (
    PROJECT_ROOT
    / "results"
    / "bdd100k.yaml"
)

if not DATA_YAML.exists():
    raise FileNotFoundError(
        f"Dataset YAML not found:\n{DATA_YAML}"
    )

print("\nDataset YAML:")
print(DATA_YAML)
print("✓ Dataset YAML found")

MODEL_NAME = "yolo26n.pt"

print("\n" + "-" * 75)
print("MODEL")
print("-" * 75)

print("Model:", MODEL_NAME)

model = YOLO(MODEL_NAME)

print("\n" + "-" * 75)
print("GPU CHECK")
print("-" * 75)

if not torch.cuda.is_available():

    raise RuntimeError(
        "\nCUDA GPU is not available.\n"
        "Please check your PyTorch/CUDA installation."
    )

DEVICE = 0

print("CUDA available :", torch.cuda.is_available())
print("GPU             :", torch.cuda.get_device_name(DEVICE))

EPOCHS = 30
PATIENCE = 10
IMAGE_SIZE = 640
BATCH_SIZE = 16
WORKERS = 4
RUN_NAME = (
    "bdd100k_original_yolo_yolo26n_baseline_30ep"
)

print("\n" + "-" * 75)
print("TRAINING CONFIGURATION")
print("-" * 75)
print("Dataset        :", DATA_YAML)
print("Model          :", MODEL_NAME)
print("Maximum epochs :", EPOCHS)
print("Patience       :", PATIENCE)
print("Image size     :", IMAGE_SIZE)
print("Batch size     :", BATCH_SIZE)
print("Device         :", DEVICE)
print("Workers        :", WORKERS)
print("Validation     : YES")
print("Pretrained     : YES")
print("Seed           : 0")
print("Deterministic  : YES")
print("\n" + "=" * 75)
print("STARTING YOLO26n BASELINE TRAINING")
print("=" * 75)
print("\nThis experiment will run:")
print(f"  Maximum epochs = {EPOCHS}")
print(f"  Early stopping = {PATIENCE} epochs")
print("\nThe model will NOT be modified.")
print("No IQA.")
print("No enhancement.")
print("No LightGBM.")
print("No adaptive preprocessing.")
print("=" * 75)


results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    patience=PATIENCE,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=WORKERS,
    project=str(BASELINE_RUNS),
    name=RUN_NAME,

    # Prevent accidental overwrite
    exist_ok=False,
    pretrained=True,
    seed=0,
    deterministic=True,
    val=True,
    save=True,
    cache=False,
    close_mosaic=10,
    verbose=True
)

RUN_DIR = (
    BASELINE_RUNS
    / RUN_NAME
)
BEST_MODEL = (
    RUN_DIR
    / "weights"
    / "best.pt"
)
LAST_MODEL = (
    RUN_DIR
    / "weights"
    / "last.pt"
)
RESULTS_CSV = (
    RUN_DIR
    / "results.csv"
)


print("\n" + "=" * 75)
print("BASELINE TRAINING FINISHED")
print("=" * 75)
print("\nRun directory:")
print(RUN_DIR)
print("\nCheckpoint status:")
print("-" * 75)
print(
    "best.pt :",
    "FOUND" if BEST_MODEL.exists() else "MISSING"
)
print(
    "last.pt :",
    "FOUND" if LAST_MODEL.exists() else "MISSING"
)
print(
    "results.csv :",
    "FOUND" if RESULTS_CSV.exists() else "MISSING"
)

if BEST_MODEL.exists():

    print("\nBest model:")
    print(BEST_MODEL)
if LAST_MODEL.exists():
    print("\nLast model:")
    print(LAST_MODEL)
if RESULTS_CSV.exists():
    print("\nTraining results:")
    print(RESULTS_CSV)
print("\n" + "=" * 75)
if (
    BEST_MODEL.exists()
    and LAST_MODEL.exists()
    and RESULTS_CSV.exists()
):
    print("STATUS: PASS")
    print(
        "YOLO26n 30-epoch / patience-10 "
        "baseline completed successfully."
    )
else:
    print("STATUS: CHECK REQUIRED")
    print("One or more expected output files are missing.")

print("=" * 75)

YOLO26n BASELINE TRAINING

Project root:
E:\ML_Project

Dataset:
E:\ML_Project\datasets\bdd100k_original_yolo

Results directory:
E:\ML_Project\results\yolo26_baseline

---------------------------------------------------------------------------
DATASET CHECK
---------------------------------------------------------------------------
Dataset directory : FOUND

Dataset YAML:
E:\ML_Project\results\bdd100k.yaml
✓ Dataset YAML found

---------------------------------------------------------------------------
MODEL
---------------------------------------------------------------------------
Model: yolo26n.pt

---------------------------------------------------------------------------
GPU CHECK
---------------------------------------------------------------------------
CUDA available : True
GPU             : NVIDIA GeForce RTX 5060 Ti

---------------------------------------------------------------------------
TRAINING CONFIGURATION
-------------------------------------------------------------

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



       5/30      5.42G      2.171      1.377   0.003874        564        640: 100% ━━━━━━━━━━━━ 4375/4375 5.3it/s 13:42<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 313/313 8.3it/s 37.8s<0.1s
                   all      10000     185526      0.607      0.346       0.37      0.196

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       6/30      5.42G       2.15      1.351    0.00381        448        640: 100% ━━━━━━━━━━━━ 4375/4375 5.4it/s 13:36<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 313/313 8.2it/s 38.1s<0.1s
                   all      10000     185526      0.606      0.359      0.382      0.205

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       7/30      5.42G      2.136      1.333   0.003759        475        640: 100% ━━━━━━━━━━━━ 4375/4375 5.3it/s 13:42<0.2s
                